# Nonlinear additive-noise models (CAM and RESIT)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/functional.ipynb)

CAM, the causal additive model, learns a DAG from data in which every variable is a nonlinear
function of its parents plus independent noise, `x_j = f_j(parents) + e_j`. When the functions are
nonlinear, the direction of every edge is identifiable, so CAM returns a DAG instead of the
equivalence class that PC and GES return. It fits a smooth function of each parent, adds edges
greedily by the likelihood, and drops the parents whose smooth terms are not significant. It also
returns a causal order.

RESIT, regression with subsequent independence tests, fits the same kind of model with noise of
any distribution, for a handful of variables.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.12.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.12.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.viz
from andrey.data import sample_scm
from andrey.metrics import shd

## Data

`sample_scm` draws six variables on an Erdos-Renyi graph, each the sum of a nonlinear function of
its parents and Gaussian noise, with 1,000 rows and standardized columns. The columns come in an
order drawn from the seed, so their order says nothing about the causal order.

In [3]:
def draw(seed: int, d: int = 6, noise: str = "gaussian"):
    """Draw 1,000 rows of d variables with additive noise; return the data and the true DAG."""
    sample = sample_scm(
        graph="erdos_renyi",
        functional="additive_noise",
        noise=noise,
        d=d,
        n=1000,
        seed=seed,
        scale="standardize",
        density=2.0,
    )
    return sample.data, sample.graph


X, truth = draw(seed=0)
print(f"{X.shape[0]} observations of {X.shape[1]} variables")
display(andrey.viz.draw(truth))

1000 observations of 6 variables


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 258" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="258" fill="#fafbfc"/><line x1="309.10" y1="134.32" x2="44.95" y2="158.46" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.05,133.41 308.64,129.34 309.55,139.30" fill="#3a4658" stroke-linejoin="round"/><line x1="310.11" y1="139.29" x2="244.37" y2="160.49" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.63,136.22 308.58,134.53 311.65,144.05" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="159.96" x2="209.01" y2="163.93" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,164.17 209.13,158.93 208.89,168.93" fill="#3a4658" stroke-linejoin="round"/><line x1="42.84" y1="166.82" x2="112.82" y2="213.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,218.60 115.57,208.92 110.06,217.26" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="32.23" x2="45.00" y2="32.23" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,32.23 109.00,27.23 109.00,37.23" fill="#3a4658" stroke-linejoin="round"/><line x1="212.39" y1="176.50" x2="143.08" y2="218.98" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.92,171.28 209.78,172.24 215.00,180.77" fill="#3a4658" stroke-linejoin="round"/><circle cx="332.00" cy="132.23" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="132.23" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="32.00" cy="159.65" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="159.65" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="132.00" cy="32.23" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="32.23" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="232.00" cy="164.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="164.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="132.00" cy="225.77" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="225.77" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="32.00" cy="32.23" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="32.23" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">5</text></svg>'

## Fit CAM

CAM returns a DAG and `ordering`, a causal order as column indices, causes first: the order of its
search before pruning.

In [4]:
cam = andrey.cam(X)
print(f"causal order {cam.ordering}")
for source, target, _ in cam.structure.oriented_edges():
    print(f"  {source} -> {target}")
display(andrey.viz.draw(cam.structure))

causal order (5, 2, 4, 3, 0, 1)
  0 -> 1
  3 -> 0
  3 -> 1
  4 -> 1
  5 -> 2
  4 -> 3


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="240" fill="#fafbfc"/><line x1="245.00" y1="186.13" x2="309.00" y2="186.13" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,186.13 309.00,181.13 309.00,191.13" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="186.13" x2="145.00" y2="186.13" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,186.13 209.00,181.13 209.00,191.13" fill="#3a4658" stroke-linejoin="round"/><path d="M 310.57 177.77 Q 232.00 147.13 144.11 181.41" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.89,181.41 312.39,173.12 308.76,182.43" fill="#3a4658" stroke-linejoin="round"/><path d="M 309.85 179.94 Q 184.64 144.94 44.97 154.59" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.48,182.63 311.19,175.12 308.50,184.75" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="53.87" x2="45.00" y2="53.87" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,53.87 109.00,48.87 109.00,58.87" fill="#3a4658" stroke-linejoin="round"/><line x1="110.01" y1="179.39" x2="44.43" y2="159.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.57,182.32 111.47,174.61 108.54,184.17" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="186.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="186.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="332.00" cy="186.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="186.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="132.00" cy="53.87" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="53.87" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="132.00" cy="186.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="186.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="32.00" cy="155.49" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="155.49" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="32.00" cy="53.87" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="53.87" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">5</text></svg>'

## Compare with the truth

The structural Hamming distance (SHD) counts the edges to add, remove, or reverse to reach the true
graph; the empty graph's SHD is the number of true edges.

In [5]:
rows = {
    "CAM": {
        "edges": len(cam.structure.oriented_edges()),
        "SHD": shd(cam.structure, truth),
    },
    "empty graph": {"edges": 0, "SHD": len(truth.oriented_edges())},
}
pd.DataFrame(rows).T

,edges,SHD
CAM,6,3
empty graph,0,6


## Check over ten datasets

The same model, drawn with ten seeds. The table gives CAM's SHD, with the empty graph's beside it.

In [6]:
rows = {}
for seed in range(10):
    data, graph = draw(seed)
    rows[seed] = {
        "CAM": shd(andrey.cam(data).structure, graph),
        "empty graph": len(graph.oriented_edges()),
    }
table = pd.DataFrame(rows).T
table.index.name = "seed"
display(table)
print("median SHD:", table.median().to_dict())

,CAM,empty graph
seed,,
0,3,6
1,0,6
2,0,6
3,0,6
4,1,6
5,0,6
6,2,6
7,3,6
8,0,6


median SHD: {'CAM': 0.5, 'empty graph': 6.0}


## Rescaling changes nothing

CAM does not depend on a column's scale or location, so multiplying or shifting a column leaves its
graph unchanged. Here every column is rescaled and shifted differently.

In [7]:
scale = np.array([0.01, 1.0, 5.0, 50.0, 300.0, 1e4])
unit_free = X * scale + 7 * scale
same = np.array_equal(andrey.cam(unit_free).structure.to_numpy(), cam.structure.to_numpy())
print(f"same graph on the rescaled data: {same}")

same graph on the rescaled data: True


## RESIT

RESIT fits the same model with noise of any distribution. It finds a causal order from the sink up:
the next sink is the variable whose additive-model residual an independence test finds most
independent of the remaining variables. Then it removes each parent whose removal leaves the residual
independent. Each independence test reads all the remaining variables at once, so RESIT suits a
handful of variables: its accuracy claim holds up to 10, and it takes seconds at 5 and minutes at 30.
Here it runs on five variables with Gumbel noise.

In [ ]:
X5, truth5 = draw(seed=0, d=5, noise="gumbel")
resit = andrey.resit(X5)
print(f"causal order {resit.ordering}")
for source, target, _ in resit.structure.oriented_edges():
    print(f"  {source} -> {target}")
print(f"SHD {shd(resit.structure, truth5)}, empty graph {len(truth5.oriented_edges())}")

## Check RESIT over ten datasets

The same model with ten seeds: RESIT's SHD, with the empty graph's beside it.

In [ ]:
rows = {}
for seed in range(10):
    data, graph = draw(seed, d=5, noise="gumbel")
    rows[seed] = {
        "RESIT": shd(andrey.resit(data).structure, graph),
        "empty graph": len(graph.oriented_edges()),
    }
table = pd.DataFrame(rows).T
table.index.name = "seed"
display(table)
print("median SHD:", table.median().to_dict())

## When to use it

CAM needs effects that are nonlinear and additive in the parents, with additive noise and no hidden
common causes. With linear effects and Gaussian noise the data identify only the equivalence class
and the direction of the other edges is arbitrary, so use PC or GES; noise whose scale depends on
the parents can reverse edges. The [0.13.0 release card](https://andrey.abel.ai/docs/benchmarks/0.13.0.html#cam)
compares CAM with CAM 1.0 from 10 to 100 variables. Change `seed`, `d`, or the noise (`"gumbel"`)
and rerun to see how it responds.

RESIT needs the same kind of data with noise of any distribution, and no more than about ten
variables: from 20 variables its errors come near the empty graph's on Gaussian noise and a fit
takes about a minute, so use CAM for more variables. The release card compares RESIT with the
authors' code and with lingam from 5 to 20 variables.